<a href="https://colab.research.google.com/github/ashuyad/learning-transformers/blob/main/notebooks/creating_your_own_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install requests

In [16]:
import requests
url = "https://api.github.com/repos/huggingface/datasets/issues?page=1&per_page=1"
response = requests.get(url)
response.status_code

200

In [ ]:
response.json()

In [17]:
import time
import math
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
import requests

def fetch_issues(
    owner="huggingface",
    repo="datasets",
    num_issues=4_000,
    rate_limit=5_000,
    issues_path=Path("."),
):
    if not issues_path.is_dir():
        issues_path.mkdir(exist_ok=True)

    batch = []
    all_issues = []
    per_page = 100  # Number of issues to return per page
    num_pages = math.ceil(num_issues / per_page)
    base_url = "https://api.github.com/repos"

    # GitHub API pagination starts at page 1
    for page in tqdm(range(1, num_pages + 1)):
        query = f"issues?page={page}&per_page={per_page}&state=all"
        response = requests.get(f"{base_url}/{owner}/{repo}/{query}", headers=headers)

        if response.status_code != 200:
            print(f"\nError fetching page {page}: {response.status_code} - {response.text}")
            break

        issues = response.json()
        if isinstance(issues, list):
            batch.extend(issues)
        else:
            print(f"\nUnexpected response structure: {issues}")
            break

        if len(batch) > rate_limit and len(all_issues) < num_issues:
            all_issues.extend(batch)
            batch = []  # Flush batch for next time period
            print(f"Reached GitHub rate limit. Sleeping for one hour ...")
            time.sleep(60 * 60 + 1)

    all_issues.extend(batch)
    if all_issues:
        df = pd.DataFrame.from_records(all_issues)
        dest_file = issues_path / f"{repo}-issues.jsonl"
        df.to_json(dest_file, orient="records", lines=True)
        print(
            f"\nDownloaded {len(all_issues)} issues for {repo}! Dataset stored at {dest_file}"
        )
    else:
        print("\nNo issues were downloaded.")

In [18]:
fetch_issues()

  0%|          | 0/40 [00:00<?, ?it/s]


Downloaded 4000 issues for datasets! Dataset stored at datasets-issues.jsonl


In [19]:
from datasets import load_dataset
issues_dataset = load_dataset("json", data_files="datasets-issues.jsonl", split="train")
issues_dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment'],
    num_rows: 4000
})

In [23]:
issues_dataset = issues_dataset.map(
    lambda x: {"is_pull_request": False if x["pull_request"] is None else True}
)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [31]:
def safe_get_comments(issue_number):
    url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        if isinstance(data, list):
            return [r["body"] for r in data if isinstance(r, dict) and "body" in r]
    return []

# Map the dataset using the safe function
issues_with_comments_dataset = issues_dataset.map(
    lambda x: {"comments": safe_get_comments(x["number"])}
)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [32]:
issues_dataset_copy = issues_dataset.filter(
    lambda x: (x["is_pull_request"] == False and x["comments"] > 0)
)
issues_dataset_copy

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment', 'is_pull_request'],
    num_rows: 1401
})

In [34]:
from huggingface_hub import notebook_login
notebook_login()

In [35]:
issues_with_comments_dataset.push_to_hub("github-issues")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmp6_trq_hr.parquet    :  14%|#4        | 3.59MB / 25.4MB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/datasets/ashuyad1234/github-issues/commit/4e712b6e60b616303c8dac84cecf5a9c0b964606', commit_message='Upload dataset', commit_description='', oid='4e712b6e60b616303c8dac84cecf5a9c0b964606', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ashuyad1234/github-issues', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ashuyad1234/github-issues'), pr_revision=None, pr_num=None)